# Visium mouse brain: processed data

Writes `processed_data/visium_brain_adata.h5ad`, the Visium mouse brain section read by `Fig2`, `Fig3`, `ExtendedDataFig2`, `ExtendedDataFig3_ExtendedDataFig4a-b`, `ExtendedDataFig3c_signatures`, `ExtendedDataFig5e-f`, `ExtendedDataFig6`, `ExtendedDataFig7` and `REV_GSD_base_brain`.

The section is `squidpy.datasets.visium_hne_adata()`. Raw counts are kept in `layers["counts"]`; `X` and `layers["log1p_norm"]` hold library-size-normalised (10,000 counts per spot), log1p-transformed counts. This reproduces the file used for the published figures exactly (`X`, both layers, spots, genes, clusters and coordinates).

In [ ]:
import os

import scanpy as sc
import squidpy as sq

In [ ]:
# repository root (the folder containing `notebooks/`); safe to re-run this cell
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
os.chdir(project_dir)

processed_data = project_dir + "processed_data/"
os.makedirs(processed_data, exist_ok=True)

In [ ]:
# downloaded once and cached in data/anndata/
adata = sq.datasets.visium_hne_adata(path=project_dir + "data/anndata/visium_hne_adata.h5ad")

# raw counts, in the gene order of X
counts = adata.raw.to_adata()[:, adata.var_names]
adata.layers["counts"] = counts.X.copy()

# library-size normalisation and log1p
norm = counts.copy()
sc.pp.normalize_total(norm, target_sum=1e4)
sc.pp.log1p(norm)
adata.X = norm.X.copy()
adata.layers["log1p_norm"] = norm.X.copy()
adata

In [ ]:
adata.write_h5ad(processed_data + "visium_brain_adata.h5ad")